In [ ]:
import pandas as pd
from datetime import date, datetime
from bs4 import BeautifulSoup
import requests
import time
from tqdm import tqdm
from pathlib import Path

## UFC Events

### 1. Events till UFC-308

In [ ]:
script_dir=Path.cwd()
file_path=script_dir.parent/"historical_data"/"ufc_events.csv"

In [ ]:
df = pd.read_csv(file_path)
df.head()

In [ ]:
df.info()

In [ ]:
urls_1 = list(df['url_link'])
len(urls_1)

### 2. Events from UFC-309

In [ ]:
urls= [
    'http://www.ufcstats.com/event-details/daff32bc96d1eabf',
    'http://www.ufcstats.com/event-details/e955046551f8c7dd',
    'http://www.ufcstats.com/event-details/ad23903ef3af7406',
    'http://www.ufcstats.com/event-details/72c9c2eadfc3277e',
    'http://www.ufcstats.com/event-details/81ddc98fceb30086',
    'http://www.ufcstats.com/event-details/39f68882def7a507',
    'http://www.ufcstats.com/event-details/80dbeb1dd5b53e64',
    'http://www.ufcstats.com/event-details/e6015889f50075d2',
    'http://www.ufcstats.com/event-details/ce7871949b0ed2bf',
    'http://www.ufcstats.com/event-details/c50ea21fe9ef478d',
    'http://www.ufcstats.com/event-details/9766ed20b68c7c33',
    'http://www.ufcstats.com/event-details/ca936c67687789e9',
    'http://www.ufcstats.com/event-details/39f62b833e4cf126',
    'http://www.ufcstats.com/event-details/cc2ad11b1f9d818b',
    'http://www.ufcstats.com/event-details/0f7210aa8d61af8d',
    'http://www.ufcstats.com/event-details/86b30a86664cb6e4',
    'http://www.ufcstats.com/event-details/22f4b6cb6b1bd7fd',
    'http://www.ufcstats.com/event-details/b2e3aca4cd363477',
    'http://www.ufcstats.com/event-details/de20ffb3fc2e7629',
    'http://www.ufcstats.com/event-details/118463dd8db16e7f',
    'http://www.ufcstats.com/event-details/8ad022dd81224f61',
    'http://www.ufcstats.com/event-details/2a898bf9fb7710b3',
    'http://www.ufcstats.com/event-details/18c49685296c60e6',
    'http://www.ufcstats.com/event-details/de277a4abcfeea46',
    'http://www.ufcstats.com/event-details/400c7b43c86d27d3',
    'http://www.ufcstats.com/event-details/66766984842c13dc',
    'http://www.ufcstats.com/event-details/7b03d9df5910917d',
    'http://www.ufcstats.com/event-details/b8e2f10efb6eca85',
    'http://www.ufcstats.com/event-details/28d8638ea0a71908',
    'http://www.ufcstats.com/event-details/f2c934689243fe4e',
    'http://www.ufcstats.com/event-details/6cd3dfc54f01287f',
    'http://www.ufcstats.com/event-details/421ccfc6ddb17958',
    'http://www.ufcstats.com/event-details/8fbcd82bf7f352bf',
    'http://www.ufcstats.com/event-details/754968e325d6f60d',
    'http://www.ufcstats.com/event-details/6e380a4d73ab4f0e',
    'http://www.ufcstats.com/event-details/5efaaf313b652dd7',
    'http://www.ufcstats.com/event-details/ed220e87bef0cc64',
    'http://www.ufcstats.com/event-details/8944a0f9b2f0ce6d',
    'http://www.ufcstats.com/event-details/9336e86cfd4ceaa1',
    'http://www.ufcstats.com/event-details/38e5d9dcb0fddc42',
    'http://www.ufcstats.com/event-details/7956f026e2672c47',
    'http://www.ufcstats.com/event-details/0e2c2daf11b5d8f2',
    'http://www.ufcstats.com/event-details/6436029b50a9c255',
    'http://www.ufcstats.com/event-details/8db1b36dde268ef6',
    'http://www.ufcstats.com/event-details/92c96df8bdab5fea',
    'http://www.ufcstats.com/event-details/bd92cf5da5413d2a',
    'http://www.ufcstats.com/event-details/bc0f994de0521926',
    'http://www.ufcstats.com/event-details/00e11b5c8b7bfeeb',
    'http://www.ufcstats.com/event-details/30ad2050273d016a',
    'http://www.ufcstats.com/event-details/c337c3c85b1871e0',
    'http://www.ufcstats.com/event-details/79ab17db3b40831a',
    'http://www.ufcstats.com/event-details/0cfbbfa0ba6d9855',
    'http://www.ufcstats.com/event-details/15ec018d144710db',
    'http://www.ufcstats.com/event-details/babc6b5745335f18',
]

In [ ]:
len(urls)

### 3. Total urls

In [ ]:
urls.extend(urls_1)
len(urls)

### Batch extract time

In [ ]:
batch_extract_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

### Extracting data from urls

#### via threadpooling

In [ ]:
import concurrent.futures

def scrap_event_data(url):
    events_fight_data = []

    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

    try:
        response = requests.get(url, headers=headers, timeout=10)
        # response = session.get(url, timeout=5)
        soup = BeautifulSoup(response.text, 'html.parser')

        event_name = soup.find('h2', class_='b-content__title').text.strip()

        events_venue = soup.find_all('li', class_='b-list__box-list-item')
        event_date, event_location = [li.text.strip().split('\n')[-1].strip() for li in events_venue]

        rows = soup.find_all('tr', class_='b-fight-details__table-row')

        for row in rows[1:]:
            cols = row.find_all('td', class_='b-fight-details__table-col')

            # cols[0] --> unwanted data
            fighter_1, fighter_2 = [p.text.strip() for p in cols[1].find_all('p')]
            result = cols[1].find_all('p')[0].text.strip()

            # knockdown
            kd_1, kd_2 = [p.text.strip() for p in cols[2].find_all('p')]
            kd = f"{kd_1}-{kd_2}"

            # strikes
            str_1, str_2 = [p.text.strip() for p in cols[3].find_all('p')]
            strikes = f"{str_1}-{str_2}"

            # takedown
            td_1, td_2 = [p.text.strip() for p in cols[4].find_all('p')]
            td = f"{td_1}-{td_2}"

            # submission
            sub_1, sub_2 = [p.text.strip() for p in cols[5].find_all('p')]
            sub = f"{sub_1}-{sub_2}"

            weight_class = cols[6].find('p').text.strip()
            method_list = [method.text.strip() for method in cols[7].find_all('p')]
            method = "-".join(method_list)

            round_num = cols[8].find('p').text.strip()
            fight_time = cols[9].find('p').text.strip()

            event_data = {
                'event_name':event_name,
                'event_date':event_date,
                'event_location':event_location,
                'event_url':url,
                'fighter_1':fighter_1,
                'fighter_2':fighter_2,
                'result':result,
                'kd':kd,
                'strikes':strikes,
                'td':td,
                'sub':sub,
                'weight_class':weight_class,
                'method':method,
                'round_num':round_num,
                'fight_time':fight_time,
                'extracted_at':batch_extract_time
            }
            
            events_fight_data.append(event_data)

    except Exception as e:
        print(f"\nFailed on {url}: {e}")
        pass

    return events_fight_data

In [ ]:
workers = 8

all_scraped_event_data = []

with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as executor:
    results = list(
        tqdm(
            executor.map(scrap_event_data, urls),
            total=len(urls),
            desc="Scarping UFC events"
        )
    )

for event_result in results:
    all_scraped_event_data.extend(event_result)

In [ ]:
events_df = pd.DataFrame(all_scraped_event_data)
print(f"Scraping complete..Total fights extracted: {len(events_df)}")

In [ ]:
events_df.head(5)

In [ ]:
events_df.info()

In [ ]:
events_df['event_date'] = pd.to_datetime(events_df['event_date'])
events_df = events_df.sort_values(by='event_date', ascending=False).reset_index(drop=True)

In [ ]:
events_df.head(5)

In [ ]:
output_file_path=script_dir.parent/"data"/"historical_ufc_events_scraped.csv"

events_df.to_csv(output_file_path, index=False)

## All UFC Fighters

### Scrap fighter data via threadpool

In [ ]:
import concurrent.futures

def scrap_fighter_data(fighter_url):

    all_fighters_data = []

    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

    try:
        response = requests.get(fighter_url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')

        rows = soup.find_all(class_='b-statistics__table-row')

        for row in rows[2:]:
            cols = row.find_all(class_='b-statistics__table-col')

            first = cols[0].text.strip()
            last = cols[1].text.strip()
            nickname = cols[2].text.strip()
            a_tag = cols[0].find('a')
            fighter_url = a_tag.get('href') if a_tag else "No URL"
            ht = cols[3].text.strip()
            wt = cols[4].text.strip()
            reach = cols[5].text.strip()
            stance = cols[6].text.strip()
            w = cols[7].text.strip()
            l = cols[8].text.strip()
            d = cols[9].text.strip()
            belt = cols[10].text.strip()
            belt_img = cols[10].find('img')
            belt = "Yes" if belt_img else "No"

            fighter_data = {
                'first_name':first,
                'last_name':last,
                'nickname':nickname,
                'fighter_url':fighter_url,
                'height':ht,
                'weight':wt,
                'reach':reach,
                'stance':stance,
                'win':w,
                'lose':l,
                'draw':d,
                'belt':belt,
                'extracted_at':batch_extract_time
            }

            all_fighters_data.append(fighter_data)

    except Exception as e:
        print(f"\nFailed on {fighter_url}: {e}")
        pass

    return all_fighters_data

In [ ]:
from string import ascii_lowercase

fighter_urls = [f"http://www.ufcstats.com/statistics/fighters?char={letter}&page=all" for letter in ascii_lowercase]

# len(fighter_urls)

workers = 8

all_scraped_fighter_data = []

with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as executor:
    results = list(
        tqdm(
            executor.map(scrap_fighter_data, fighter_urls),
            total=len(fighter_urls),
            desc="Scarping UFC fighters"
        )
    )

for fighter_result in results:
    all_scraped_fighter_data.extend(fighter_result)


In [ ]:
fighters_df = pd.DataFrame(all_scraped_fighter_data)

fighters_df.head()

In [ ]:
print(f"Total UFC fighters extracted: {len(fighters_df)}")

In [ ]:
output_file_path=script_dir.parent/"data"/"fighters_data_scraped_test.csv"
fighters_df.to_csv(output_file_path, index=False)

### Data Testing

In [ ]:
import pandas as pd

events_df = pd.read_csv('historical_data/final_ufc_events_scraped.csv')
fighters_df = pd.read_csv('historical_data/fighters_data_scraped.csv')

In [ ]:
events_df

In [ ]:
events_df[events_df['method'].str.endswith('-')]

In [ ]:
x=datetime.now()

In [ ]:
x

In [ ]:
x.time()